# 05 — Evaluation

## What this notebook does
1. Loads the best checkpoint (lowest validation loss)
2. Evaluates on UQA validation set (greedy + beam)
3. Evaluates on Wiki-UQA (greedy + beam)
4. Computes BLEU-4, ROUGE-L, perplexity, unk rate
5. Saves automatic_metrics.csv and samples.tsv
6. Creates human evaluation template

## Why Wiki-UQA?
Wiki-UQA is an out-of-domain evaluation set. Performance degradation
on Wiki-UQA compared to UQA reveals how well the model generalises
beyond its training data.

## Perplexity note
Perplexity = exp(mean cross-entropy) from teacher-forced evaluation.
It does NOT depend on greedy vs beam decoding — it measures how well
the model predicts the reference targets.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from configs.config import *
from src.tokenizer.tokenizer_utils import UrduTokenizer
from src.data.dataset import QGenDataset, collate_fn
from src.model.encoder import Encoder
from src.model.decoder import Decoder
from src.model.seq2seq import Seq2Seq
from src.training.utils import set_seed, get_device, load_checkpoint
from src.training.train import validate_epoch
from src.evaluation.evaluate import evaluate_model, save_automatic_metrics, save_samples
from src.evaluation.human_eval import create_evaluation_template

In [ ]:
set_seed(SEED)
device = get_device()

tokenizer = UrduTokenizer(SP_MODEL_PATH)
vocab_size = tokenizer.vocab_size

In [ ]:
# Build and load model
enc_hidden = HIDDEN_SIZE * 2
encoder = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, PAD_ID)
decoder = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, enc_hidden, NUM_LAYERS, DROPOUT, PAD_ID)
model = Seq2Seq(encoder, decoder).to(device)

load_checkpoint(BEST_MODEL_PATH, model, device=device)

## UQA Validation Evaluation

In [ ]:
valid_dataset = QGenDataset(VALID_FILE, tokenizer)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
val_loss = validate_epoch(model, valid_loader, criterion, device)
print(f'UQA validation loss: {val_loss:.4f}')

uqa_metrics, uqa_gen = evaluate_model(
    model, tokenizer, valid_dataset, device, val_loss,
    dataset_label='UQA-valid', beam_width=BEAM_WIDTH,
)

for method, m in uqa_metrics.items():
    print(f"  {m['dataset']} + {m['decoding']}: BLEU={m['bleu4']}, ROUGE-L={m['rouge_l']}, PPL={m['perplexity']}, UNK={m['unk_rate']}")

## Wiki-UQA Evaluation

In [ ]:
wiki_path = Path(WIKI_VALID_FILE)
all_metrics = [uqa_metrics['greedy'], uqa_metrics['beam']]

if wiki_path.exists():
    wiki_dataset = QGenDataset(wiki_path, tokenizer)
    wiki_loader = DataLoader(wiki_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    wiki_loss = validate_epoch(model, wiki_loader, criterion, device)
    print(f'Wiki-UQA loss: {wiki_loss:.4f}')
    
    wiki_metrics, _ = evaluate_model(
        model, tokenizer, wiki_dataset, device, wiki_loss,
        dataset_label='Wiki-UQA', beam_width=BEAM_WIDTH,
    )
    all_metrics.extend([wiki_metrics['greedy'], wiki_metrics['beam']])
    
    for method, m in wiki_metrics.items():
        print(f"  {m['dataset']} + {m['decoding']}: BLEU={m['bleu4']}, ROUGE-L={m['rouge_l']}, PPL={m['perplexity']}, UNK={m['unk_rate']}")
else:
    print(f'Wiki-UQA file not found at {wiki_path}, skipping.')

## Save results

In [ ]:
# Save automatic metrics table
save_automatic_metrics(all_metrics, RESULTS_DIR / 'automatic_metrics.csv')

# Save 50 sample outputs
save_samples(uqa_gen, RESULTS_DIR / 'samples.tsv')

# Create human evaluation template
create_evaluation_template(RESULTS_DIR / 'samples.tsv')

In [ ]:
# Display metrics table
import pandas as pd
df = pd.DataFrame(all_metrics)
df